# Example 1.3.1 / 1: Integer Variables

Companion notebook for **1.3.1 Integer, Binary and Restricted Variables** in the AMPL lab Documentation, section "Integer Variables". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Integer Variables

The integer optimum is not simply the continuous optimum rounded. In this small model the
continuous solution is `x = 3, y = 1.5`; rounding gives `x = 3, y = 2`, which breaks the second
constraint, or `x = 3, y = 1`, which is worse than the true integer optimum.

In `run.py`, `ampl` is an `AMPL` object that has already read `model.mod` (this model needs no
data, so there is no `data.py`). `ampl.option["relax_integrality"] = 1` tells AMPL to ignore
`integer` and solve the continuous problem; setting it back to 0 gives the integer optimum.
`value()` reads each variable and the objective:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
var x integer >= 0;
var y integer >= 0;
maximize Value: 5*x + 4*y;
subject to Weight: 6*x + 4*y <= 24;
subject to Space:    x + 2*y <= 6;

Writing model.mod


### amplpy: run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.read("model.mod")

In [4]:
# run.py
ampl.option["relax_integrality"] = 1   # ignore "integer": the continuous optimum
ampl.solve(solver="highs")
print(ampl.var["x"].value(), ampl.var["y"].value(), ampl.obj["Value"].value())
ampl.option["relax_integrality"] = 0   # the integer optimum
ampl.solve(solver="highs")
print(ampl.var["x"].value(), ampl.var["y"].value(), ampl.obj["Value"].value())

HiGHS 1.14.0: optimal solution; objective 21
2 simplex iterations
0 barrier iterations
3.0 1.5 21.0
HiGHS 1.14.0: optimal solution; objective 20
0 simplex iterations
1 branching nodes
4.0 0.0 20.0


### Pure AMPL: model.run

The same in pure AMPL, with the same `model.mod`; `option relax_integrality` is set with the
`option` command:

In [5]:
%%ampl_eval
reset;
model model.mod;

In [6]:
%%ampl_eval
# model.run
option solver highs;
option relax_integrality 1;   # ignore "integer": the continuous optimum
solve;
display x, y, Value;
option relax_integrality 0;   # the integer optimum
solve;
display x, y, Value;

HiGHS 1.14.0: optimal solution; objective 21
2 simplex iterations
0 barrier iterations
x = 3
y = 1.5
Value = 21

HiGHS 1.14.0: optimal solution; objective 20
0 simplex iterations
1 branching nodes
x = 4
y = 0
Value = 20

